In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
# If someone is reading this and is not just throwing this through unit tests,
# I do want to apologize for the lack of comments, I've had to juggle studying with work and my brain is too fried to explain.
# I've kept most things within the scope of the course. I've also relied on the labs quite a bit
path = path + '/Q1_data.csv'
df = pd.read_csv(path)

print("Dataset loaded successfully into DataFrame 'df' from the /content/ directory.")

In [ ]:
display(df.head())


In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.histplot(df['Delivery_Time'], kde=True, bins=50)
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df = df.drop('Order_ID', axis=1)

display(df.head())

In [ ]:

print("Missing values before handling:")
display(df.isnull().sum())

for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    if df[col].isnull().any():
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value)

if df['Courier_Experience_yrs'].isnull().any():
    median_value = df['Courier_Experience_yrs'].median()
    df['Courier_Experience_yrs'] = df['Courier_Experience_yrs'].fillna(median_value)

initial_rows = len(df)
df = df.dropna(subset=['Delivery_Time'])
dropped_rows = initial_rows - len(df)


display(df.isnull().sum())

In [ ]:
duplicates_before = df.duplicated().sum()

if duplicates_before > 0:
    df.drop_duplicates(inplace=True)
    duplicates_after = df.duplicated().sum()
    print('Dupes removed')
else:
    print("No duper")

In [ ]:
#I was going to use one-hot but, honestly, get_dummy feels more intuitive to me.
categorical_cols = df.select_dtypes(include='object').columns
if len(categorical_cols) > 0:
    df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

display(df.head())

In [ ]:
from sklearn.preprocessing import StandardScaler

X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

X_scaled.head()

In [ ]:
X = X_scaled

In [ ]:
# Task 2,3,4,5: Write your code here:

from sklearn.model_selection import KFold

kf = KFold(n_splits=6, shuffle=True, random_state=21)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model = RandomForestRegressor(random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

average_mae = np.mean(mae_scores)

print(f"MAE for each fold: {mae_scores}")
print(f"Averaged Mean Absolute Error across all folds: {average_mae:.4f}")

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns

feature_importances = model.feature_importances_
features = X.columns
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)
plt.figure()
sns.barplot(x='Importance', y='Feature', data=importance_df)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
plt.figure(figsize=(10, 6))
sns.histplot(y_pred, kde=True, bins=30, color = 'red') #wanted a red colour for this one :)
plt.show()